# MLflow Model Registry

This notebook demonstrates how to interact with the MLflow model registry to
browse, compare, and promote trained models.

## Prerequisites
- MLflow tracking server accessible (configured via `MLFLOW_TRACKING_URI` env var)
- Completed training runs logged to MLflow (Phase 06)

In [ ]:
import os
import mlflow
from mlflow.tracking import MlflowClient

# Configure MLflow
mlflow.set_tracking_uri(
    os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow.mlflow.svc.cluster.local:5000")
)

client = MlflowClient()
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

## Browse Registered Models

In [ ]:
# List all registered models
registered_models = client.search_registered_models()

print(f"Found {len(registered_models)} registered model(s):\n")
for model in registered_models:
    latest = model.latest_versions
    versions_info = ", ".join(
        [f"v{v.version} ({v.current_stage})" for v in latest]
    )
    print(f"  {model.name}")
    print(f"    Description: {model.description or 'N/A'}")
    print(f"    Versions:    {versions_info}")
    print()

## Compare Model Versions

Look at metrics across different versions of a model.

In [ ]:
import pandas as pd

# Select a model name (use the first registered model or specify one)
model_name = registered_models[0].name if registered_models else "isaac-cartpole-ppo"

# Get all versions for this model
versions = client.search_model_versions(f"name='{model_name}'")

rows = []
for v in versions:
    run = client.get_run(v.run_id)
    metrics = run.data.metrics
    params = run.data.params
    rows.append({
        "version": v.version,
        "stage": v.current_stage,
        "run_id": v.run_id[:8],
        "final_reward": metrics.get("final_reward", None),
        "max_reward": metrics.get("max_reward", None),
        "training_iterations": metrics.get("total_iterations", None),
        "algorithm": params.get("algorithm", "N/A"),
        "learning_rate": params.get("learning_rate", "N/A"),
    })

df_versions = pd.DataFrame(rows)
print(f"Model: {model_name}")
df_versions

## Promote a Model to Production

Transition the best model version to the "Production" stage.

In [ ]:
# Find the version with the highest final reward
if len(df_versions) > 0:
    best_row = df_versions.loc[df_versions["final_reward"].idxmax()]
    best_version = best_row["version"]

    print(f"Best version: v{best_version} (final_reward={best_row['final_reward']})")
    print()

    # Uncomment to promote to Production:
    # client.transition_model_version_stage(
    #     name=model_name,
    #     version=best_version,
    #     stage="Production",
    #     archive_existing_versions=True,
    # )
    # print(f"Model {model_name} v{best_version} promoted to Production")
    print("To promote, uncomment the transition_model_version_stage call above.")
else:
    print("No model versions available.")

## Load and Inspect a Model

In [ ]:
# Load the production model (if one exists)
try:
    production_model = mlflow.pyfunc.load_model(
        model_uri=f"models:/{model_name}/Production"
    )
    print(f"Loaded production model: {model_name}")
    print(f"  Model type: {type(production_model)}")
    print(f"  Metadata:   {production_model.metadata}")
except Exception as e:
    print(f"No production model available: {e}")
    print("Run the promotion cell above first, or load a specific version:")
    print(f"  mlflow.pyfunc.load_model('models:/{model_name}/1')")